# Corrected FADC3D encoder — seed 42

**Branch:** `feature/fadc3d-correct`

Discrete voxelwise 3D frequency-adaptive dilation with SHARED kernels.

- One learnable base kernel per adaptive convolution.
- Dilations exactly `(1,1,1) / (2,2,2) / (3,3,3)`, per-voxel `k_att` softmax over dim=1.
- Kernel-side AdaKern3D attention on the `W_low + W_high` decomposition.
- `FrequencySelection3D` on the input feature map, defaults `k_list=[2,4,8]`.
- Corrected FADC only in `enc1..enc4`; bottleneck + decoder use plain Conv3d.

Deliberately not claimed: continuous AdaDR, deformable Conv3d, or any Dice number.

**Hardened notebook** — every substantive cell aborts loudly (SystemExit) if
its prerequisite fails, so a stale checkout or failed test never proceeds to
100-epoch training silently.


In [ ]:
# ── CONFIG ─────────────────────────────────────────────────────────────
SEED                = 42
GIT_BRANCH          = "feature/fadc3d-correct"

DATA_ROOT              = "/kaggle/input/datasets/bharathvemurik/mama-mia-preprocessed-cache-2ch"
PREPROCESSED_CACHE_DIR = DATA_ROOT
CODE_DIR               = "/kaggle/working/FADC-3D"

OUTPUT_DIR_SMOKE  = "/kaggle/working/outputs/fadc3d_correct_encoder_smoke"
OUTPUT_DIR_FULL   = "/kaggle/working/outputs/fadc3d_correct_encoder_s42"

# Full-training hyperparameters
EPOCHS         = 100
BATCH_SIZE     = 2
NUM_WORKERS    = 4
PATCH_SIZE     = [128, 128, 64]
LEARNING_RATE  = 1e-4
WARMUP_EPOCHS  = 5
VAL_EVERY      = 10
VAL_OVERLAP    = 0.5
DEEP_SUPERVISION = True

# k_att schedule
K_ATT_TEMP_START    = 2.0
K_ATT_TEMP_END      = 1.0
K_ATT_ANNEAL_EPOCHS = 60

# Attention diversity aux DISABLED for this run.
ATTN_DIVERSITY_WEIGHT = 0.0

# Smoke-test parameters (separate from full training).
SMOKE_PATCH_SIZE = [48, 48, 24]

MODEL_NAME = "unet3d_fadc_encoder_correct"

print(f"SEED                : {SEED}")
print(f"BRANCH              : {GIT_BRANCH}")
print(f"MODEL_NAME          : {MODEL_NAME}")
print(f"OUTPUT_DIR_FULL     : {OUTPUT_DIR_FULL}")
print(f"OUTPUT_DIR_SMOKE    : {OUTPUT_DIR_SMOKE}")
print(f"PATCH_SIZE (train)  : {PATCH_SIZE}")
print(f"PATCH_SIZE (smoke)  : {SMOKE_PATCH_SIZE}")
print(f"EPOCHS              : {EPOCHS}")
print(f"BATCH_SIZE          : {BATCH_SIZE}")
print(f"VAL_EVERY           : {VAL_EVERY}   VAL_OVERLAP: {VAL_OVERLAP}")
print(f"k_att T             : {K_ATT_TEMP_START} -> {K_ATT_TEMP_END} over {K_ATT_ANNEAL_EPOCHS} ep")
print(f"attn_diversity_wt   : {ATTN_DIVERSITY_WEIGHT}  (disabled)")


In [ ]:
# ── 1. INSTALL DEPS + REQUIRE CUDA ─────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "monai",
                "--upgrade-strategy", "only-if-needed", "-q"], check=True)

import torch
print(f"PyTorch        : {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise SystemExit(
        "CUDA is not available on this session. This notebook is designed "
        "to run on a GPU-backed Kaggle kernel — switch the accelerator "
        "to GPU T4 x2 (or equivalent) and re-run."
    )
print(f"GPU            : {torch.cuda.get_device_name(0)}")
p = torch.cuda.get_device_properties(0)
print(f"VRAM           : {p.total_memory / 1e9:.1f} GB")


In [ ]:
# ── 2. CLONE / CHECKOUT feature/fadc3d-correct (subprocess, check=True) ─
import os, sys, subprocess

def _run(cmd, cwd=None):
    """subprocess.run wrapper that surfaces stderr and check=True."""
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        sys.stdout.write(r.stdout)
        sys.stderr.write(r.stderr)
        raise SystemExit(f"Command failed ({r.returncode}): {' '.join(cmd)}")
    return r.stdout.strip()

if os.path.exists(CODE_DIR):
    print(f"repo present; fetching {GIT_BRANCH} ...")
    _run(["git", "-C", CODE_DIR, "fetch", "--all"])
    _run(["git", "-C", CODE_DIR, "checkout", GIT_BRANCH])
    _run(["git", "-C", CODE_DIR, "pull", "--ff-only"])
else:
    _run(["git", "clone", "-b", GIT_BRANCH,
          "https://github.com/Vemuri-BK/FADC-3D.git", CODE_DIR])

sys.path.insert(0, CODE_DIR)

# Verify active branch matches GIT_BRANCH and capture the exact commit hash.
active = _run(["git", "-C", CODE_DIR, "rev-parse", "--abbrev-ref", "HEAD"])
if active != GIT_BRANCH:
    raise SystemExit(f"Active branch is {active!r}, expected {GIT_BRANCH!r}. "
                     "Refusing to run on the wrong branch.")
GIT_COMMIT_HASH = _run(["git", "-C", CODE_DIR, "rev-parse", "HEAD"])
GIT_COMMIT_LINE = _run(["git", "-C", CODE_DIR, "log", "-1", "--oneline"])
print(f"branch  : {active}")
print(f"HEAD    : {GIT_COMMIT_LINE}")
print(f"commit  : {GIT_COMMIT_HASH}")

# Persist the commit hash next to full-training outputs so a downloaded
# checkpoint can be traced back to its source commit later.
os.makedirs(OUTPUT_DIR_FULL, exist_ok=True)
with open(os.path.join(OUTPUT_DIR_FULL, "source_commit.txt"), "w", encoding="utf-8") as f:
    f.write(GIT_COMMIT_HASH + "\n" + GIT_COMMIT_LINE + "\n")

for p in ("fadc_3d_correct/adaptive_dilated_conv_3d.py",
          "fadc_3d_correct/ada_kernel_3d.py",
          "fadc_3d_correct/freq_select_3d.py",
          "models/unet_3d_fadc_correct.py",
          "training/train_centralized_correct.py",
          "tests/test_fadc_3d_correct.py",
          "diag_fadc_3d_correct.py"):
    assert os.path.exists(os.path.join(CODE_DIR, p)), f"missing on branch: {p}"
print("Corrected FADC3D files present on branch.")


In [ ]:
# ── 3. PULL VERIFIER + ARCHITECTURE CHECKS ─────────────────────────────
# Aborts BEFORE training if the corrected code did not land.
import inspect, sys
sys.path.insert(0, CODE_DIR)

from fadc_3d_correct.adaptive_dilated_conv_3d import AdaptiveDilatedConv3D
from fadc_3d_correct.ada_kernel_3d import AdaKern3D
from fadc_3d_correct.freq_select_3d import FrequencySelection3D
from models.unet_3d_fadc_correct import (
    build_unet3d_fadc_correct, EXPECTED_ADAPTIVE_CONV_COUNT, MODEL_NAMES,
)

# 1) module signatures
assert AdaptiveDilatedConv3D.KERNEL_SIZE == 3
assert MODEL_NAME in MODEL_NAMES, f"MODEL_NAME {MODEL_NAME} not in {MODEL_NAMES}"

# 2) build fresh encoder-only model
m = build_unet3d_fadc_correct(MODEL_NAME, in_channels=2, out_channels=2,
                              base_filters=32, deep_supervision=DEEP_SUPERVISION).cuda()

# 3) EXACTLY 8 adaptive convs, all under enc*
n_adapt = m.count_adaptive_convs()
assert n_adapt == EXPECTED_ADAPTIVE_CONV_COUNT["encoder"] == 8, \
    f"encoder placement must yield 8 adaptive convs, got {n_adapt}"
adapt_names = m.adaptive_conv_names()
outside_enc = [n for n in adapt_names if not n.startswith("enc")]
assert not outside_enc, f"adaptive convs found outside enc*: {outside_enc}"

# 4) sanity: single base kernel per adaptive conv
for name, mod in m.named_modules():
    if isinstance(mod, AdaptiveDilatedConv3D):
        top_weights = [n for n, p in mod.named_parameters(recurse=False) if n == "weight"]
        assert top_weights == ["weight"], f"{name} has unexpected top-level kernel params: {top_weights}"

# 5) dilation list is (1, 2, 3)
sample = next(mm for mm in m.modules() if isinstance(mm, AdaptiveDilatedConv3D))
assert sample.dilation_list == (1, 2, 3), sample.dilation_list

print(f"MODEL_NAME       : {MODEL_NAME}")
print(f"adaptive convs   : {n_adapt}/8 (all under enc*)")
print(f"params           : {sum(p.numel() for p in m.parameters()):,}")
print("Pull + architecture checks passed.")
del m


In [ ]:
# ── 4. RUN CORRECTNESS TESTS (abort on nonzero) ───────────────────────
import subprocess, sys
res = subprocess.run(
    [sys.executable, "-m", "tests.test_fadc_3d_correct"],
    cwd=CODE_DIR, capture_output=True, text=True,
)
print(res.stdout[-6000:])
if res.returncode != 0:
    sys.stderr.write(res.stderr[-3000:])
    raise SystemExit(f"Correctness tests FAILED (exit {res.returncode}) — refusing to launch training.")
print("Correctness tests PASSED.")


In [ ]:
# ── 5. SMOKE TRAINING — 2 ep on 4 cases, small patch (abort on nonzero) ─
import os, subprocess, sys

train_script = os.path.join(CODE_DIR, "training", "train_centralized_correct.py")
os.makedirs(OUTPUT_DIR_SMOKE, exist_ok=True)

cmd = [
    sys.executable, "-u", train_script,
    "--model",         MODEL_NAME,
    "--data_root",     DATA_ROOT,
    "--output_dir",    OUTPUT_DIR_SMOKE,
    "--patch_size",    str(SMOKE_PATCH_SIZE[0]), str(SMOKE_PATCH_SIZE[1]), str(SMOKE_PATCH_SIZE[2]),
    "--batch_size",    "2",
    "--warmup_epochs", "1",
    "--val_every",     "1",
    "--val_overlap",   str(VAL_OVERLAP),
    "--seed",          str(SEED),
    "--k_att_temp_start", str(K_ATT_TEMP_START),
    "--k_att_temp_end",   str(K_ATT_TEMP_END),
    "--k_att_anneal_epochs", "1",
    "--preprocessed_cache_dir", PREPROCESSED_CACHE_DIR,
    "--smoke_test",
]
if DEEP_SUPERVISION:
    cmd.append("--deep_supervision")

print("SMOKE command:\n  " + " ".join(cmd))
print("=" * 60, flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
while True:
    chunk = proc.stdout.read(512)
    if not chunk: break
    sys.stdout.write(chunk.decode("utf-8", errors="replace"))
    sys.stdout.flush()
proc.wait()
print(f"\nSMOKE exit code: {proc.returncode}")
if proc.returncode != 0:
    raise SystemExit(f"SMOKE training failed (exit {proc.returncode}). Refusing to launch full training.")


In [ ]:
# ── 6. SMOKE CKPT: strict=True reload + forward parity ────────────────
import os, torch, sys
sys.path.insert(0, CODE_DIR)
from models.unet_3d_fadc_correct import build_unet3d_fadc_correct

ckpt_path = os.path.join(OUTPUT_DIR_SMOKE, "last_checkpoint.pth")
assert os.path.exists(ckpt_path), f"smoke checkpoint missing: {ckpt_path}"
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
print(f"epoch    : {ckpt.get('epoch')}")
print(f"best_dice: {ckpt.get('best_dice')}")
arch = ckpt.get("arch_identity")
print(f"arch_id  : {arch}")

model = build_unet3d_fadc_correct(
    arch["model_name"],
    in_channels=arch["in_channels"], out_channels=arch["out_channels"],
    base_filters=arch["base_filters"], deep_supervision=arch["deep_supervision"],
).eval()
missing, unexpected = model.load_state_dict(ckpt["model"], strict=True)
print(f"strict load: missing={len(missing)} unexpected={len(unexpected)}")
assert not missing and not unexpected, "strict reload mismatch"

x = torch.randn(1, 2, 32, 32, 16)
with torch.no_grad():
    y1 = model(x)
    y2 = model(x)
diff = (y1 - y2).abs().max().item()
assert diff < 1e-6, f"non-deterministic forward under eval(): {diff}"
print(f"forward parity  max|y1-y2| = {diff:.2e}  OK")


In [ ]:
# ── 7. GPU MEMORY PROBE — one 128x128x64 batch, INCLUDING every DS head ─
# If this OOMs we STOP. We do NOT silently shrink the patch or batch size.
import torch, gc, os, sys
sys.path.insert(0, CODE_DIR)
from models.unet_3d_fadc_correct import build_unet3d_fadc_correct

torch.cuda.empty_cache(); gc.collect()
torch.cuda.reset_peak_memory_stats()
print(f"pre-probe VRAM alloc: {torch.cuda.memory_allocated()/1e9:.2f} GB")

model = build_unet3d_fadc_correct(
    MODEL_NAME, in_channels=2, out_channels=2, base_filters=32,
    deep_supervision=DEEP_SUPERVISION,
).cuda().train()

from torch.amp import autocast, GradScaler
scaler = GradScaler("cuda")
try:
    x = torch.randn(BATCH_SIZE, 2, *PATCH_SIZE, device="cuda")
    with autocast("cuda"):
        y = model(x)
        # Deep supervision: y is a tuple (main, ds2, ds3, ds4) when training.
        # Include EVERY output in the probe loss so gradients flow through
        # the auxiliary heads too.
        outputs = y if isinstance(y, tuple) else (y,)
        loss = sum(o.float().pow(2).mean() for o in outputs)
    scaler.scale(loss).backward()
    peak = torch.cuda.max_memory_allocated() / 1e9
    print(f"probe OK. n_outputs_in_loss={len(outputs)} "
          f"peak VRAM alloc: {peak:.2f} GB")
except RuntimeError as e:
    if "out of memory" in str(e).lower():
        peak = torch.cuda.max_memory_allocated() / 1e9
        print(f"OOM at probe. peak alloc: {peak:.2f} GB")
        raise SystemExit(
            "Full-size probe OOMed. Not launching full training. "
            "Do NOT silently change patch_size or batch_size — the "
            "experiment contract requires the declared PATCH_SIZE/BATCH_SIZE."
        )
    raise
finally:
    del model
    torch.cuda.empty_cache(); gc.collect()


In [ ]:
# ── 8. FULL TRAINING — 100 ep (abort on nonzero exit) ─────────────────
import os, subprocess, sys

train_script = os.path.join(CODE_DIR, "training", "train_centralized_correct.py")
os.makedirs(OUTPUT_DIR_FULL, exist_ok=True)

cmd = [
    sys.executable, "-u", train_script,
    "--model",              MODEL_NAME,
    "--data_root",          DATA_ROOT,
    "--output_dir",         OUTPUT_DIR_FULL,
    "--epochs",             str(EPOCHS),
    "--batch_size",         str(BATCH_SIZE),
    "--num_workers",        str(NUM_WORKERS),
    "--patch_size",         str(PATCH_SIZE[0]), str(PATCH_SIZE[1]), str(PATCH_SIZE[2]),
    "--lr",                 str(LEARNING_RATE),
    "--warmup_epochs",      str(WARMUP_EPOCHS),
    "--val_every",          str(VAL_EVERY),
    "--val_overlap",        str(VAL_OVERLAP),
    "--seed",               str(SEED),
    "--k_att_temp_start",   str(K_ATT_TEMP_START),
    "--k_att_temp_end",     str(K_ATT_TEMP_END),
    "--k_att_anneal_epochs",str(K_ATT_ANNEAL_EPOCHS),
    "--preprocessed_cache_dir", PREPROCESSED_CACHE_DIR,
]
if DEEP_SUPERVISION:
    cmd.append("--deep_supervision")

print("FULL command:\n  " + " ".join(cmd))
print("=" * 60, flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
while True:
    chunk = proc.stdout.read(512)
    if not chunk: break
    sys.stdout.write(chunk.decode("utf-8", errors="replace"))
    sys.stdout.flush()
proc.wait()
print(f"\nFULL exit code: {proc.returncode}")
if proc.returncode != 0:
    raise SystemExit(f"FULL training exited nonzero ({proc.returncode}). Downstream diagnostic skipped.")


In [ ]:
# ── 9. REAL-MRI DIAGNOSTIC ON best_model.pth ──────────────────────────
import os, subprocess, sys
best_ckpt = os.path.join(OUTPUT_DIR_FULL, "best_model.pth")
val_cache = os.path.join(PREPROCESSED_CACHE_DIR, "val")
if not os.path.exists(best_ckpt):
    print(f"no best checkpoint at {best_ckpt}; skipping diagnostic.")
else:
    cmd = [
        sys.executable, os.path.join(CODE_DIR, "diag_fadc_3d_correct.py"),
        "--ckpt", best_ckpt,
        "--preprocessed_cache", val_cache,
        "--n_patches", "4",
        "--patch_size", "96", "96", "48",
    ]
    print("DIAG command:\n  " + " ".join(cmd))
    print("=" * 60, flush=True)
    subprocess.run(cmd, check=False)


In [ ]:
# ── 10. DOWNLOAD LINKS ────────────────────────────────────────────────
import os
from IPython.display import FileLink, display
for fname in ("best_model.pth", "last_checkpoint.pth",
              "train_log.json", "meta.json", "source_commit.txt"):
    p = os.path.join(OUTPUT_DIR_FULL, fname)
    if os.path.exists(p):
        print(fname); display(FileLink(p))
    else:
        print(f"(missing) {fname}")
